In [1]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="bloom_train.json")

/Users/lukasio/FinalYearProject/qtrain/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
bt_map = {
    "BT1": "Remembering",
    "BT2": "Understanding",
    "BT3": "Applying",
    "BT4": "Analyzing",
    "BT5": "Evaluating",
    "BT6": "Creating"
}

import json

with open("bloom_raw.json", "r") as f:
    data = json.load(f)

converted = []
for item in data:
    converted.append({
        "question": item["Questions"],
        "bloom_level": bt_map.get(item["Category"], "Unknown")
    })

with open("bloom_train.json", "w") as f:
    json.dump(converted, f, indent=2)

In [3]:
import json
from tqdm import tqdm
from mlx_lm import load, generate

# Load your dataset
with open("bloom_train.json", "r") as f:
    data = json.load(f)

# Choose a supported MLX model (TinyLlama is fast and good for this)
model_name = "mlx-community/TinyLlama-1.1B-Chat-v1.0-4bit"
model, tokenizer = load(model_name)

# Prompt template
def build_prompt(question):
    return f"Extract the core topic of the following question:\n\n{question}\n\nTopic:"

# Generate topics
for item in tqdm(data):
    question = item["question"]
    prompt = build_prompt(question)

    output = generate(model, tokenizer, prompt, max_tokens=16)
    topic = output.strip().split("\n")[0]
    item["topic"] = topic

# Save result
with open("bloom_train_with_topics_sample.json", "w") as f:
    json.dump(data, f, indent=2)

print("✅ Done! Topics added to first 100 and saved to bloom_train_with_topics_sample.json")

100%|██████████| 8767/8767 [21:27<00:00,  6.81it/s]

✅ Done! Topics added to first 100 and saved to bloom_train_with_topics_sample.json


In [5]:
from transformers import AutoTokenizer

# Load tokenizer
model_id = "NousResearch/Hermes-3-Llama-3.1-8B"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# Define prompt format and tokenize input + target
def tokenize_example(example):
    prompt = f"### Bloom Level: {example['bloom_level']}\n### Topic: {example['topic']}\n### Question:"
    target = example["question"]

    tokenized = tokenizer(
        prompt,
        text_target=target,
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors="pt"
    )

    return {
        "input_ids": tokenized["input_ids"][0],
        "attention_mask": tokenized["attention_mask"][0],
        "labels": tokenized["labels"][0]
    }


# Tokenize
tokenized_dataset = dataset["train"].map(tokenize_example)

Map: 100%|██████████| 8767/8767 [00:04<00:00, 2121.19 examples/s]


In [9]:
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM

# Load the base model (Hermes-3 is based on LLaMA-3.1)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    device_map="auto"  # or "cpu" if you don't have a GPU
)

# LoRA config
lora_config = LoraConfig(
    r=8,                          # Rank of LoRA matrices
    lora_alpha=32,                # Scaling factor
    lora_dropout=0.05,            # Dropout for LoRA layers
    bias="none",                  # Don’t tune biases
    task_type="CAUSAL_LM"         # We're fine-tuning a language model
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# Print to confirm
model.print_trainable_parameters()

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]Error while downloading from https://cdn-lfs-us-1.hf.co/repos/e8/dc/e8dcf99c21959bfbb15c4d77e4e4814373a33c74189698733f8749043d78885d/52ae38eb93ba324e7f6cdc0e23479c94312c39ac4e8f3fa4c76eab2d6cdb5447?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model-00003-of-00004.safetensors%3B+filename%3D%22model-00003-of-00004.safetensors%22%3B&Expires=1750079053&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc1MDA3OTA1M319LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy11cy0xLmhmLmNvL3JlcG9zL2U4L2RjL2U4ZGNmOTljMjE5NTliZmJiMTVjNGQ3N2U0ZTQ4MTQzNzNhMzNjNzQxODk2OTg3MzNmODc0OTA0M2Q3ODg4NWQvNTJhZTM4ZWI5M2JhMzI0ZTdmNmNkYzBlMjM0NzljOTQzMTJjMzlhYzRlOGYzZmE0Yzc2ZWFiMmQ2Y2RiNTQ0Nz9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSoifV19&Signature=idPzLuWKBY8zBze-Wsre21ggWxjhTm7bQrgvyfj7scNy1Soc%7EwKgwS68fAvDJPBG-40jJtzvJCp58E8AsAKlFN4RkZA9Y303lprV99sZB93igAHEyZAhCIUddfexSyAioH5rDTaVtUzo7FmO5Rs7sTGYjlQV6rQEm-7Q2uZudOfe1

ConnectionError: (MaxRetryError('HTTPSConnectionPool(host=\'cdn-lfs-us-1.hf.co\', port=443): Max retries exceeded with url: /repos/e8/dc/e8dcf99c21959bfbb15c4d77e4e4814373a33c74189698733f8749043d78885d/43eeed1bc0e0898d4525170720ab01a08f68c57df348f7af61adff9ffb6a96ff?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model-00001-of-00004.safetensors%3B+filename%3D%22model-00001-of-00004.safetensors%22%3B&Expires=1750079053&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc1MDA3OTA1M319LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy11cy0xLmhmLmNvL3JlcG9zL2U4L2RjL2U4ZGNmOTljMjE5NTliZmJiMTVjNGQ3N2U0ZTQ4MTQzNzNhMzNjNzQxODk2OTg3MzNmODc0OTA0M2Q3ODg4NWQvNDNlZWVkMWJjMGUwODk4ZDQ1MjUxNzA3MjBhYjAxYTA4ZjY4YzU3ZGYzNDhmN2FmNjFhZGZmOWZmYjZhOTZmZj9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSoifV19&Signature=FT1A1vbYUAdHjSkq~RyQnEXTfLmjUn7dzZu8-UuCOwlPu66PMx21kQiiU0Xkn6qMqZqrF6TNhY9XCCH0R0c-AyPQTX1X0Lm-tAzfmlIWVeK280WxLrLldkdwa7puykD54If3g5svVeIdhAuj0~WiQnucBwy9~52Yz

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Set training parameters
training_args = TrainingArguments(
    output_dir="./hermes-bloom-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,    # Effective batch size = 4 × 4 = 16
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=20,
    save_strategy="epoch",
    fp16=False,  # Set True if using GPU with float16 support
    push_to_hub=False
)

# Data collator for causal LM (language modeling)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # We're doing Causal LM, not Masked LM
)

from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    device_map="auto",
    load_in_8bit=True  # optional: saves VRAM, needs bitsandbytes
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

# Start training
trainer.train()